# Lab 9: Run delayed action chunks on virtual ticks

**Goal:** Learn how a delayed policy reply reaches the robot controller. Use 50 ms virtual ticks. Check chunk availability, the index `k - t`, delays of 0, 1, 2, and 4 ticks, jitter, bad replies, underflow, and reset. Then measure wall clock latency as a separate quantity.

**Inputs:** Synthetic `ActionChunk` objects only. **Compute:** CPU. **Cloud cost:** US$0. Start Jupyter from the repository root. Read the [lab guide](README.md).

**Limit:** This lab does not run SmolVLA, LIBERO, or RTC. The chunks are made by hand. The checks prove scheduler rules. They do not show robot task success.

## 1. Set up the virtual clock

One virtual tick is one controller step of 50 ms. A delay `d` is an integer number of ticks. Store `d`. Calculate milliseconds from the tick period.

Symbols in this lab:

| Symbol | Meaning |
|---|---|
| `t` | Tick of the observation that made the request |
| `d` | Delay in ticks from observation to availability |
| `t + d` | Availability tick of the reply |
| `k` | Current controller tick |
| `k - t` | Index of the action to use from a chunk |
| `H` | Number of actions in a chunk |
| `s` | Ticks between request starts |

Each synthetic action has x value `0.05 * (index + 1)`. The index is easy to read from the value. The gripper value is -1.

In [ ]:
import time

import numpy as np

from vla_lab.contracts import ActionChunk, ActionSchema
from vla_lab.runtime.timing import measure_calls
from vla_lab.runtime.virtual_scheduler import CONTROL_PERIOD_MS, VirtualScheduler, latency_to_ticks

EPISODE = "lab9-episode"
TASK = "synthetic_task"
H = 8


def make_chunk(origin, delay, request, horizon=H, reset=0, episode=EPISODE):
    actions = np.zeros((horizon, 7), dtype=np.float32)
    actions[:, 0] = 0.05 * (np.arange(horizon) + 1)
    actions[:, 6] = -1.0
    return ActionChunk(
        episode_id=episode,
        reset_counter=reset,
        request_id=request,
        observation_tick=origin,
        availability_tick=origin + delay,
        actions=actions,
        schema=ActionSchema.libero_relative(),
        controller_period_ms=CONTROL_PERIOD_MS,
    )


def x_value(index):
    return 0.05 * (index + 1)


def run(scheduler, ticks, **step_args):
    return [scheduler.step(tick, **step_args) for tick in range(ticks)]


def show(rows):
    print("tick  x      grip  underflow  age  depth  accepted / rejected")
    for row in rows:
        age = "-" if row.observation_age_ticks is None else row.observation_age_ticks
        print(
            f"{row.tick:>4}  {row.action[0]:.2f}  {row.action[6]:>4.0f}  {str(row.underflow):<9}  "
            f"{age!s:>3}  {row.queue_depth:>5}  {list(row.accepted)} / {list(row.rejected)}"
        )


assert CONTROL_PERIOD_MS == 50
print("Delay in ticks -> milliseconds:", {d: d * CONTROL_PERIOD_MS for d in (0, 1, 2, 4)})

## 2. Check availability and the index `k - t`

Send one chunk at `t = 0`. Give it delay `d`. Run 10 ticks. The rules are:

1. Before tick `t + d`, the reply is not available. The scheduler holds.
2. From tick `t + d`, the controller uses index `k - t`.
3. After tick `t + H - 1`, the chunk has no more actions.

A delayed chunk does not start at index 0. The first `d` actions are already in the past.

In [ ]:
for d in (0, 1, 2, 4):
    scheduler = VirtualScheduler(EPISODE, 0, TASK)
    scheduler.submit(make_chunk(0, d, f"d{d}"))
    rows = run(scheduler, 10)
    for row in rows:
        if d <= row.tick < H:
            assert not row.underflow
            assert row.action[0] == np.float32(x_value(row.tick - 0))  # index is k - t
            assert row.observation_age_ticks == row.tick
        else:
            assert row.underflow
            assert row.action[0] == 0.0
    first = rows[d]
    assert first.action[0] != np.float32(x_value(0)) or d == 0  # not restarted at index 0
    print(f"d={d} ({d * CONTROL_PERIOD_MS} ms): first used index {d}, x={first.action[0]:.2f}")

scheduler = VirtualScheduler(EPISODE, 0, TASK)
scheduler.submit(make_chunk(0, 2, "show"))
show(run(scheduler, 10))

## 3. Overlap chunks and find the startup gap

A real runner asks for a new chunk every `s` ticks. Use `H = 8` and `s = 4`. Send the request at `t = 0, 4, 8, ...`. The scheduler uses the newest accepted chunk that still has an action for the tick. The first `d` ticks have no chunk. After that, the overlap removes gaps.

A valid delay setting for this lab has `d <= s <= H - d`. The function below is a local teaching check. It is not the RTC validator.

In [ ]:
def valid_timing(delay, stride, horizon):
    return delay <= stride <= horizon - delay


assert valid_timing(0, 4, 8) and valid_timing(2, 4, 8) and valid_timing(4, 4, 8)
assert not valid_timing(4, 1, 8)  # s=1, d=4 is invalid
assert not valid_timing(5, 4, 8)  # s > H - d

S, TOTAL = 4, 20
for d in (0, 1, 2, 4):
    scheduler = VirtualScheduler(EPISODE, 0, TASK)
    for n, origin in enumerate(range(0, TOTAL, S)):
        scheduler.submit(make_chunk(origin, d, f"r{n}"))
    rows = run(scheduler, TOTAL)
    underflow_ticks = [row.tick for row in rows if row.underflow]
    assert underflow_ticks == list(range(d))  # only the startup gap
    for row in rows[d:]:
        newest = max(o for o in range(0, TOTAL, S) if o + d <= row.tick)
        assert row.observation_age_ticks == row.tick - newest
        assert row.action[0] == np.float32(x_value(row.tick - newest))
    print(f"d={d}: underflow ticks {underflow_ticks}, mean age {np.mean([r.observation_age_ticks for r in rows[d:]]):.2f} ticks")

## 4. Add jitter

Jitter means that the delay changes from request to request. Use a fixed list. Do not use a random number here. Then every learner gets the same run.

The delays below are `[1, 3, 0, 2, 4]` for requests at `t = 0, 2, 4, 6, 8`. The availability ticks are 1, 5, 4, 8, and 12. Request 1 has tick 5. Request 2 has tick 4. Request 2 arrives before request 1. Request 1 is older, so the scheduler rejects it as `reordered`.

You can also convert measured milliseconds to ticks. The rule is ceiling to the next tick. This is a conservative choice.

In [ ]:
assert [latency_to_ticks(ms) for ms in (0, 12, 50, 55, 130, 200)] == [0, 1, 1, 2, 3, 4]

delays = [1, 3, 0, 2, 4]
origins = [0, 2, 4, 6, 8]
scheduler = VirtualScheduler(EPISODE, 0, TASK)
for n, (origin, d) in enumerate(zip(origins, delays)):
    scheduler.submit(make_chunk(origin, d, f"j{n}"))
rows = run(scheduler, 14)
show(rows)

accepted = [rid for row in rows for rid in row.accepted]
rejected = [(row.tick, kind) for row in rows for kind in row.rejected]
assert accepted == ["j0", "j2", "j3", "j4"]
assert rejected == [(5, "reordered")]
assert rows[4].action[0] == np.float32(x_value(0))  # j2: t=4, k=4, index 0
assert rows[5].action[0] == np.float32(x_value(1))  # j2 stays newest; j1 is rejected

## 5. Reject bad replies

The scheduler checks replies in this order:

1. `duplicate`: the request ID was seen before.
2. `reordered`: the observation tick is not newer than the newest accepted tick.
3. `exhausted`: the chunk has no action left at the current tick.

A reply from another episode or reset fails in `submit`. It raises `ValueError`.

An exhausted reply needs a policy outage. A chunk with `t = 0`, `d = 2`, and `H = 3` is valid when it is created. If the policy is unavailable during ticks 2 and 3, the reply waits. At tick 4 the reply is delivered. It has no action left, so the scheduler rejects it.

In [ ]:
# Duplicate and reordered replies.
scheduler = VirtualScheduler(EPISODE, 0, TASK)
scheduler.submit(make_chunk(1, 0, "new"))
scheduler.submit(make_chunk(1, 0, "new"))      # same request ID
scheduler.submit(make_chunk(1, 1, "twin"))     # new ID, same observation tick
scheduler.submit(make_chunk(0, 1, "old"))      # older observation tick
rows = run(scheduler, 3)
assert rows[1].accepted == ("new",)
assert rows[1].rejected == ("duplicate", "reordered")  # copy and older tick
assert rows[2].rejected == ("reordered",)             # twin arrives at tick 2
assert rows[1].action[0] == np.float32(x_value(0))
assert rows[2].action[0] == np.float32(x_value(1))
print("tick 1 accepted:", rows[1].accepted, "rejected:", rows[1].rejected)
print("tick 2 rejected:", rows[2].rejected)

# Exhausted reply after a policy outage.
scheduler = VirtualScheduler(EPISODE, 0, TASK)
scheduler.submit(make_chunk(0, 2, "late", horizon=3))
rows = []
for tick in range(6):
    rows.append(scheduler.step(tick, policy_available=tick not in (2, 3)))
assert rows[3].pending_responses == 1 and rows[3].underflow
assert rows[4].rejected == ("exhausted",) and rows[4].underflow
assert rows[4].pending_responses == 0
print("tick 4 rejected:", rows[4].rejected)

# Reply from another reset.
try:
    VirtualScheduler(EPISODE, 1, TASK).submit(make_chunk(0, 0, "stale", reset=0))
except ValueError as error:
    print("Stale reset rejected:", error)
else:
    raise AssertionError("Stale reset must fail")

## 6. Hold on underflow

Underflow means no valid chunk covers the tick. The scheduler then sends zero motion in the six arm values. It repeats the last gripper value. Before the first action, the gripper value is 0.

The synthetic chunk uses gripper -1. After the chunk ends, the hold value stays -1. This test uses the scheduler only. It does not prove how the real LIBERO controller moves under zero input. That check needs a simulator run.

The scheduler records camera state in `camera_available`. It does not change the action for a camera fault. The runner must decide how to react.

In [ ]:
scheduler = VirtualScheduler(EPISODE, 0, TASK)
scheduler.submit(make_chunk(0, 1, "hold", horizon=3))
rows = run(scheduler, 6, camera_available=True)
assert [row.underflow for row in rows] == [True, False, False, True, True, True]
assert rows[0].action[6] == 0.0               # no gripper command yet
assert all(row.action[6] == -1.0 for row in rows[1:])  # last gripper value held
assert all(np.all(row.action[:6] == 0) for row in rows[3:])  # zero arm motion
assert all(row.observation_age_ticks is None for row in rows if row.underflow)

camera_row = VirtualScheduler(EPISODE, 0, TASK).step(0, camera_available=False)
assert camera_row.camera_available is False and camera_row.underflow
show(rows)

## 7. Average two chunks

Temporal ensembling averages every active chunk at tick `k`. Each chunk gives index `k - t`. Set the decay to 0 for equal weights. The result at tick 1 is the mean of chunk A index 1 and chunk B index 0.

The averaging code is adapter code from this course. It is not the upstream ACT function. Its test uses known values.

In [ ]:
scheduler = VirtualScheduler(EPISODE, 0, TASK, ensemble_decay=0.0)
scheduler.submit(make_chunk(0, 0, "A"))
scheduler.submit(make_chunk(1, 0, "B"))
rows = run(scheduler, 3, ensemble=True)
assert rows[1].queue_depth == 2
assert rows[1].action[0] == np.float32((x_value(1) + x_value(0)) / 2)
assert rows[2].action[0] == np.float32((x_value(2) + x_value(1)) / 2)
print("tick 1 average x:", float(rows[1].action[0]))

## 8. Reset

A reset clears pending replies, accepted chunks, the gripper hold value, and the tick counter. Use a new episode ID or reset counter. Policy state is outside this scheduler. The runner must reset it at the same time.

In [ ]:
scheduler = VirtualScheduler(EPISODE, 0, TASK)
scheduler.submit(make_chunk(0, 3, "pending"))
scheduler.submit(make_chunk(0, 0, "active", horizon=4))
run(scheduler, 2)
assert scheduler.step(2).action[6] == -1.0

scheduler.reset("lab9-episode-2", 1, TASK)
first = scheduler.step(0)  # the tick counter starts again
assert first.underflow and first.pending_responses == 0 and first.queue_depth == 0
assert first.action[6] == 0.0  # old gripper hold is gone
for old in (make_chunk(0, 0, "old"), make_chunk(0, 0, "old-reset", episode="lab9-episode-2", reset=0)):
    try:
        scheduler.submit(old)
    except ValueError:
        pass
    else:
        raise AssertionError("Old reset must fail")
scheduler.submit(make_chunk(1, 0, "fresh", episode="lab9-episode-2", reset=1))
assert scheduler.step(1).accepted == ("fresh",)
print("Reset removed old pending work and old chunks.")

## 9. Measure wall clock latency apart from virtual ticks

Wall clock time is how long a real call takes on this host. A virtual delay is an integer that you set. The scheduler reads only the virtual delay. It never reads a wall clock.

First, use a fake clock. This check is exact. Then time a real CPU call. The real numbers change from host to host and from run to run, so the notebook does not assert them. Do not compare wall clocks from different hosts.

`measure_calls` keeps the first (cold) call apart from later (warm) calls. It counts deadline misses against 50 ms.

In [ ]:
# Exact check with a fake clock: durations are 80, 20, 60, 30 ms.
fake = iter((0.0, 0.08, 1.0, 1.02, 2.0, 2.06, 3.0, 3.03))
report = measure_calls(lambda i: None, count=4, deadline_ms=50, clock=lambda: next(fake))
assert report.cold_deadline_miss and report.warm_deadline_misses == 1
assert [round(v) for v in report.warm_ms] == [20, 60, 30]
# Conservative conversion of the warm p95 to a virtual delay.
assert latency_to_ticks(report.warm_p95_ms) == 2

# Real CPU timing. Values depend on this host.
matrix = np.random.default_rng(0).standard_normal((256, 256)).astype(np.float32)
real = measure_calls(lambda i: matrix @ matrix, count=21, deadline_ms=CONTROL_PERIOD_MS)
assert len(real.warm_ms) == 20 and all(v >= 0 for v in real.warm_ms)
print(f"cold {real.cold_ms:.2f} ms, warm p50 {real.warm_p50_ms:.2f} ms, p95 {real.warm_p95_ms:.2f} ms")
print("deadline misses (warm):", real.warm_deadline_misses)
print("conservative delay from p95:", latency_to_ticks(real.warm_p95_ms), "ticks")

# The virtual result is the same for any wall clock speed.
def virtual_run():
    scheduler = VirtualScheduler(EPISODE, 0, TASK)
    for n, origin in enumerate(range(0, 12, 4)):
        scheduler.submit(make_chunk(origin, 2, f"w{n}"))
    return np.stack([row.action for row in run(scheduler, 12)])

first_run = virtual_run()
time.sleep(0.05)
assert np.array_equal(first_run, virtual_run())
print("Two virtual runs are identical.")

## 10. Read the saved development settings sweep

The six conditions below used one live LIBERO state and the same SmolVLA checkpoint. The saved report records model requests, virtual underflow, success, and wall clock time. Read it as an illustration of controller settings, not a setting ranking.


In [ ]:
import json
from pathlib import Path

def find_workspace(start: Path) -> Path:
    for root in (start, *start.parents):
        if (root / "pyproject.toml").is_file() and (root / "artifacts/reference/runtime/settings_sweep.json").is_file():
            return root
    raise FileNotFoundError("Run the notebook inside the course repository")

workspace = find_workspace(Path.cwd())
sweep = json.loads((workspace / "artifacts/reference/runtime/settings_sweep.json").read_text())
expected = {"synchronous-s01-d0", "synchronous-s05-d0", "synchronous-s10-d0", "naive_async-s10-d1", "naive_async-s10-d2", "naive_async-s10-d4"}
assert {row["condition"] for row in sweep["conditions"]} == expected
assert sweep["state_id"] == 0 and sweep["partition"] == "development"
for row in sweep["conditions"]:
    print(row["condition"], "success", row["success"], "requests", row["requests"], "underflow", row["underflow"], "steps", row["steps"])


## 11. What you proved and what you did not prove

You proved these rules with synthetic chunks:

- Availability is `t + d`. The action index is `k - t`.
- Delays of 0, 1, 2, and 4 ticks give 0, 50, 100, and 200 ms.
- Duplicate, reordered, and exhausted replies are rejected. A reply from an old reset cannot enter.
- Underflow gives zero arm motion and holds the gripper value.
- Reset clears all scheduler state.
- Wall clock timing is a separate measurement.

The synthetic cells did not run a policy or simulator. The saved sweep used SmolVLA and LIBERO on one development state, but this notebook did not repeat the GPU runs. One state cannot rank controller settings. No cell tested RTC or a real network. Do not call a reduced virtual delay a measured speed gain.

Next, read the [exercise](exercise.md). Read the [solution guide](../../solutions/09-runtime.md) after you save your answer.